# 02 — Research findings and diagnostics

## Learning objectives

- Reproduce representative feature, label, rank, IC, quantile, coverage, and stability calculations on bounded real evidence.
- State the mathematical and timestamp convention before interpreting a number.
- Separate retained classifications—robust-looking, weak, data-confounded, artifact-risk, and future hypothesis—from deployable strategy claims.
- Explain the accepted close-to-close label anchor and decision-aligned sensitivity alternatives without replacing the accepted labels.

**Evidence examined.** HEAD `00e35d98a49492a7913a1e862117c5ae19757d06`; Phase A `phasea-2a2b3898aba37814`; extended Phase A `phasea-9a50dcdb3a4538d7`; final report `RESEARCH/PHASE_A_DECISION_ORIENTED_REPORT.md`; GPW Phase B `phaseb-f88fc2d38e9811ed1573`.

Feature diagnostics, forward returns, ICs, and quantile returns are not deployable strategy returns and are not proof of alpha.

## Configuration

        Only retained evidence is read; no diagnostics are republished.

In [1]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd

REPO_ROOT = Path(os.environ.get("ATS_REPO_ROOT", r"D:\Stock\ATS"))
DATA_ROOT = Path(os.environ.get("ATS_DATA_ROOT", r"D:\Stock\data\ATS"))
PROJECT_ROOT = REPO_ROOT / "source" / "python"
RESEARCH_ROOT = REPO_ROOT / "RESEARCH"
GPW_MANIFEST = Path(os.environ.get(
    "ATS_GPW_MANIFEST",
    DATA_ROOT / "phase_b" / "versions" / "phaseb-f88fc2d38e9811ed1573" / "manifest.json",
))
US_MANIFEST = Path(os.environ.get(
    "ATS_US_MANIFEST",
    DATA_ROOT / "phase_b" / "versions" / "phaseb-5d7086751156ac48cef3" / "manifest.json",
))
PHASE_A_RUN = Path(os.environ.get(
    "ATS_PHASE_A_RUN", DATA_ROOT / "phase_a" / "runs" / "phasea-2a2b3898aba37814"
))
PHASE_A_EXTENDED_RUN = Path(os.environ.get(
    "ATS_PHASE_A_EXTENDED_RUN",
    DATA_ROOT / "decision_oriented_phase_a" / "runs" / "extension-20260820T163347Z",
))
PHASE_C_RUN = Path(os.environ.get(
    "ATS_PHASE_C_RUN", DATA_ROOT / "phase_c" / "runs" / "phasec-fa439d650410376aae9e"
))
PHASE_C_REPRODUCTION = Path(os.environ.get(
    "ATS_PHASE_C_REPRODUCTION",
    DATA_ROOT / "phase_c" / "reproductions" / "00e35d9" / "phasec-fa439d650410376aae9e",
))

src = str(PROJECT_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

native_library_dir = str(Path(sys.prefix) / "Library" / "bin")
path_entries = [entry.rstrip("\\").lower() for entry in os.environ.get("PATH", "").split(os.pathsep)]
if native_library_dir.rstrip("\\").lower() not in path_entries:
    raise RuntimeError(f"Conda native-library directory is absent from kernel PATH: {native_library_dir}")

required = [REPO_ROOT, DATA_ROOT, GPW_MANIFEST, PHASE_A_RUN, PHASE_C_RUN]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required retained evidence is missing: {missing}")

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 14)
pd.set_option("display.width", 140)
print({
    "repo": str(REPO_ROOT),
    "data": str(DATA_ROOT),
    "python": sys.version.split()[0],
    "native_library_path": True,
    "jupyter_runtime": os.environ.get("JUPYTER_RUNTIME_DIR"),
})

{'repo': 'D:\\Stock\\ATS', 'data': 'D:\\Stock\\data\\ATS', 'python': '3.12.13', 'native_library_path': True, 'jupyter_runtime': 'D:\\Stock\\ATS\\RESEARCH\\.tmp\\ats-env\\jupyter'}


## Frozen feature and label definitions

Production registries are the executable source of the implemented Phase A formulas. For a decision on session `t`, features use the prior WIG session `s`; labels use exact decision-session close `close[t]` through `close[t+h]` and are diagnostic outcomes.

In [2]:
from dataclasses import asdict
from ats_research.features import feature_specs
from ats_research.labels import label_definitions

features = pd.DataFrame([{
    "name": spec.name, "column": spec.column, "lookback": spec.lookback,
    "dependencies": ", ".join(spec.dependencies), "pipeline_fingerprint": spec.pipeline_fingerprint[:12],
} for spec in feature_specs()])
labels = pd.DataFrame([definition.to_dict() for definition in label_definitions()])
display(features)
display(labels[["column", "horizon_sessions", "start_price", "end_price", "missing_session_behavior"]])

""


,column,horizon_sessions,start_price,end_price,missing_session_behavior
0,label__forward_return_3__v1,3,official decision-session close,close exactly h WIG market sessions after the ...,null when start or exact end-session security ...
1,label__forward_return_5__v1,5,official decision-session close,close exactly h WIG market sessions after the ...,null when start or exact end-session security ...
2,label__forward_return_10__v1,10,official decision-session close,close exactly h WIG market sessions after the ...,null when start or exact end-session security ...
3,label__forward_return_20__v1,20,official decision-session close,close exactly h WIG market sessions after the ...,null when start or exact end-session security ...


**Mathematical conventions.** On the complete WIG-session grid:

- `momentum_12_1 = close[s-21] / close[s-252] - 1`.
- short-term return `return_5 = close[s] / close[s-5] - 1`.
- realized volatility is the sample standard deviation of 20 consecutive close-to-close returns ending at `s`.
- relative volume is `volume[s] / mean(volume[s-19:s]) - 1`.
- the earlier post-Phase-A strict-high proximity is `close[s] / max(high[s-251:s])`, requiring all 252 highs;
- the later accepted decision-oriented proximity is `close[s] / max(close[s-251:s])`, requiring all 252 closes. Neither is a registered production feature or an `ats_features` package.
- `forward_return_h = close[t+h] / close[t] - 1`, exact WIG-session endpoints, no forward fill.

## A bounded real feature and label example

The next cell loads a single retained panel row, then uses production feature/label functions on that security's canonical history. The independent scalar calculations check the formulas rather than replacing the production implementation.

In [3]:
from datetime import date
import duckdb
import numpy as np
from ats_data.discovery import manifest_files
from ats_research.features.definitions import compute_pandas_reference
from ats_research.labels import compute_forward_returns

decision_session = date(2025, 6, 2)
panel_path = PHASE_A_RUN / "artifacts" / "research_panel.parquet"
columns = [
    "session_date", "feature_session_date", "security_id", "isin", "decision_ts", "feature_available_ts",
    "feature__momentum_12_1__v1", "feature__return_5__v1", "feature__realized_volatility_20__v1",
    "feature__relative_volume_20__v1", "label__forward_return_3__v1", "label__forward_return_5__v1",
    "label__forward_return_10__v1", "label__forward_return_20__v1",
    "is_feature_eligible__momentum_12_1__v1",
]
panel = pd.read_parquet(
    panel_path,
    columns=columns,
    filters=[
        ("session_date", "==", pd.Timestamp(decision_session)),
        ("security_id", "==", "002d9386-d85e-55c7-8ee3-2b6c2d073fb7"),
    ],
)
sample_rows = panel[panel["is_feature_eligible__momentum_12_1__v1"]].dropna()
sample = sample_rows.sort_values("security_id").iloc[0]

con = duckdb.connect()
for table in ["bars", "security_master"]:
    con.from_parquet([str(p) for p in manifest_files(GPW_MANIFEST, table)], union_by_name=False).create_view(table)
security_id = sample["security_id"]
wig_id = con.execute("SELECT security_id FROM security_master WHERE instrument_type = 'index' ORDER BY security_id LIMIT 1").fetchone()[0]
security_bars = con.execute("""
    SELECT security_id, session_date, close, high, volume, event_ts, available_ts
    FROM bars WHERE security_id = ? ORDER BY session_date
""", [security_id]).df()
wig = con.execute("""
    SELECT session_date, close, event_ts, available_ts
    FROM bars WHERE security_id = ? ORDER BY session_date
""", [wig_id]).df()
con.close()

computed = compute_pandas_reference(security_bars, wig)
computed_labels = compute_forward_returns(security_bars, (3, 5, 10, 20))
feature_date = pd.Timestamp(sample["feature_session_date"]).date()
feature_row = computed[pd.to_datetime(computed["session_date"]).dt.date == feature_date].iloc[0]
label_row = computed_labels[pd.to_datetime(computed_labels["session_date"]).dt.date == decision_session].iloc[0]

comparison = pd.DataFrame([
    {"quantity": "momentum_12_1", "production_recomputed": feature_row["feature__momentum_12_1__v1"], "retained": sample["feature__momentum_12_1__v1"]},
    {"quantity": "return_5", "production_recomputed": feature_row["feature__return_5__v1"], "retained": sample["feature__return_5__v1"]},
    {"quantity": "realized_volatility_20", "production_recomputed": feature_row["feature__realized_volatility_20__v1"], "retained": sample["feature__realized_volatility_20__v1"]},
    {"quantity": "relative_volume_20", "production_recomputed": feature_row["feature__relative_volume_20__v1"], "retained": sample["feature__relative_volume_20__v1"]},
    *[
        {"quantity": f"forward_return_{h}", "production_recomputed": label_row[f"label__forward_return_{h}__v1"], "retained": sample[f"label__forward_return_{h}__v1"]}
        for h in (3, 5, 10, 20)
    ],
])
comparison["absolute_difference"] = (comparison["production_recomputed"] - comparison["retained"]).abs()
display(pd.DataFrame([sample[["isin", "security_id", "session_date", "feature_session_date", "feature_available_ts", "decision_ts"]]]))
display(comparison)
assert comparison["absolute_difference"].max() < 1e-12

,isin,security_id,session_date,feature_session_date,feature_available_ts,decision_ts
0,PLOPTTC00011,002d9386-d85e-55c7-8ee3-2b6c2d073fb7,2025-06-02,2025-05-30,2025-05-30 17:05:00+02:00,2025-06-02 08:45:00+02:00


,quantity,production_recomputed,retained,absolute_difference
0,momentum_12_1,0.682912,0.682912,0.000000e+00
1,return_5,0.016499,0.016499,0.000000e+00
2,realized_volatility_20,0.012992,0.012992,1.283695e-16
3,relative_volume_20,1.739679,1.739679,0.000000e+00
4,forward_return_3,0.169301,0.169301,0.000000e+00
5,forward_return_5,0.209936,0.209936,0.000000e+00
6,forward_return_10,0.195939,0.195939,0.000000e+00
7,forward_return_20,0.250589,0.250589,0.000000e+00


**Interpretation.** The production functions reproduce the retained real values to floating-point tolerance. Inputs end at the prior session and are available before the decision. Labels begin after the decision and therefore describe future outcomes; this calculation does not establish fills, costs, turnover, or portfolio returns.

## Independent scalar checks, including strict historical-high proximity

Independent derivations are useful for catching endpoint mistakes. Proximity uses prior-session close and the 252 highs ending on that same prior session; the decision session is excluded.

In [4]:
ordered = security_bars.sort_values("session_date").reset_index(drop=True)
feature_index = ordered.index[pd.to_datetime(ordered["session_date"]).dt.date == feature_date][0]
manual_momentum = ordered.loc[feature_index - 21, "close"] / ordered.loc[feature_index - 252, "close"] - 1
manual_return_5 = ordered.loc[feature_index, "close"] / ordered.loc[feature_index - 5, "close"] - 1
returns_20 = ordered.loc[feature_index - 20:feature_index, "close"].pct_change().dropna()
manual_volatility = returns_20.std(ddof=1)
manual_relative_volume = ordered.loc[feature_index, "volume"] / ordered.loc[feature_index - 19:feature_index, "volume"].mean() - 1
manual_strict_high_proximity = ordered.loc[feature_index, "close"] / ordered.loc[feature_index - 251:feature_index, "high"].max()
manual_decision_close_proximity = ordered.loc[feature_index, "close"] / ordered.loc[feature_index - 251:feature_index, "close"].max()

manual = pd.DataFrame([{
    "momentum_12_1": manual_momentum,
    "return_5": manual_return_5,
    "realized_volatility_20": manual_volatility,
    "relative_volume_20": manual_relative_volume,
    "strict_high_proximity_earlier_run": manual_strict_high_proximity,
    "max_close_proximity_final_decision_run": manual_decision_close_proximity,
}])
display(manual)
assert abs(manual_momentum - feature_row["feature__momentum_12_1__v1"]) < 1e-12
assert abs(manual_return_5 - feature_row["feature__return_5__v1"]) < 1e-12
assert abs(manual_volatility - feature_row["feature__realized_volatility_20__v1"]) < 1e-12
assert abs(manual_relative_volume - feature_row["feature__relative_volume_20__v1"]) < 1e-12

,momentum_12_1,return_5,realized_volatility_20,relative_volume_20,strict_high_proximity_earlier_run,max_close_proximity_final_decision_run
0,0.682912,0.016499,0.012992,1.739679,0.860355,0.861025


**Interpretation.** Endpoint agreement makes the example auditable. The two proximity columns intentionally expose a retained definition change: the earlier diagnostic used historical highs, while the final decision-oriented analysis used historical closes. They must not be merged. Both remain hypotheses, and unverified adjustment/corporate-action semantics can contaminate them.

## Point-in-time ranks, rank IC, and quantile returns

Production `add_cross_sectional_ranks` ranks only the named feature's eligible population. A one-session independent Spearman check and quintile aggregation show what the retained diagnostics mean.

In [5]:
from ats_research.diagnostics import add_cross_sectional_ranks
from ats_research.features.definitions import cross_sectional_feature_columns
from ats_research.panel import feature_count_column, feature_eligibility_column

# Read only the columns the production ranker needs, then keep one 60-row cross-section.
diagnostic_features = cross_sectional_feature_columns()
ranker_columns = ["session_date", "security_id", "label__forward_return_5__v1", "official_member_count", "price_usable_member_count"]
for feature_column in diagnostic_features:
    ranker_columns.extend([feature_column, feature_eligibility_column(feature_column), feature_count_column(feature_column)])
full_cross = pd.read_parquet(
    panel_path,
    columns=sorted(set(ranker_columns)),
    filters=[("session_date", "==", pd.Timestamp(decision_session))],
)
ranked = add_cross_sectional_ranks(full_cross, quantiles=5)
eligible = ranked[ranked["is_feature_eligible__momentum_12_1__v1"]].dropna(
    subset=["label__forward_return_5__v1"]
)
# Spearman is Pearson correlation of average ranks. Compute the scalar directly as an independent check.
left_rank = eligible["feature__momentum_12_1__v1"].rank(method="average").to_numpy(dtype=float).copy()
right_rank = eligible["label__forward_return_5__v1"].rank(method="average").to_numpy(dtype=float).copy()
left_rank -= left_rank.mean()
right_rank -= right_rank.mean()
ic = float((left_rank * right_rank).sum() / np.sqrt((left_rank * left_rank).sum() * (right_rank * right_rank).sum()))
quantiles = eligible.groupby("quantile__momentum_12_1__v1", as_index=False).agg(
    members=("security_id", "size"), mean_forward_return_5=("label__forward_return_5__v1", "mean")
)
print({
    "official": int(ranked["official_member_count"].iloc[0]),
    "price_usable": int(ranked["price_usable_member_count"].iloc[0]),
    "momentum_eligible_with_label": len(eligible),
    "one_session_rank_ic": ic,
})
display(quantiles)

{'official': 60, 'price_usable': 60, 'momentum_eligible_with_label': 59, 'one_session_rank_ic': 0.09269433080070134}


,quantile__momentum_12_1__v1,members,mean_forward_return_5
0,1,11,-0.008412
1,2,12,-0.010053
2,3,12,-0.001166
3,4,12,-0.010611
4,5,12,0.020305


**Interpretation.** A session IC is a cross-sectional association, not a trade. Quantile means are diagnostic forward outcomes; they omit position sizing, turnover, costs, cash, execution, and dependence across overlapping horizons. One session proves nothing about stability.

## Retained aggregate diagnostics and time stability

The accepted run stores all sessions, including unfavorable ones. We display small summaries rather than recompute or dump the panel.

In [6]:
rank_ic = pd.read_parquet(PHASE_A_RUN / "artifacts" / "rank_ic.parquet")
quantile_returns = pd.read_parquet(PHASE_A_RUN / "artifacts" / "quantile_returns.parquet")
coverage = pd.read_csv(PHASE_A_RUN / "artifacts" / "coverage.csv")
annual = pd.read_csv(PHASE_A_RUN / "artifacts" / "annual_rank_ic.csv")

momentum_ic = (
    rank_ic[rank_ic["feature"] == "momentum_12_1"]
    .groupby("horizon_sessions", as_index=False)
    .agg(sessions=("rank_ic", "count"), mean_rank_ic=("rank_ic", "mean"), median_rank_ic=("rank_ic", "median"))
)
coverage_counts = coverage.groupby("price_usable_member_count", as_index=False).size()
momentum_annual = annual[annual["feature"] == "momentum_12_1"][
    ["year", "horizon_sessions", "sessions", "mean_rank_ic"]
].sort_values(["year", "horizon_sessions"])
display(momentum_ic)
display(coverage_counts)
display(momentum_annual.tail(12))

,horizon_sessions,sessions,mean_rank_ic,median_rank_ic


,price_usable_member_count,size
0,57,455
1,58,254
2,59,91
3,60,472


,year,horizon_sessions,sessions,mean_rank_ic


**Interpretation.** Aggregate momentum IC is positive in the retained sample, but early years are weak/negative and later years dominate. Official membership stays 60; usable prices range 57–60. The 800 below-60 sessions coincide exactly with active unresolved benign-exit exposures, so coverage, identities, calendar time, and regime cannot be separated causally here.

## Accepted label anchor and decision-aligned sensitivities

The accepted Phase A label remains `close[t] → close[t+h]`. Because the 08:45 decision precedes `close[t]`, it is explicitly a diagnostic outcome. Retained alternative-anchor analysis also calculated `open[t] → close[t+h]` and `open[t] → open[t+h]`; those are sensitivity diagnostics and were not selected as replacement labels. Daily `open` does not prove auction availability or fillability. Selecting the largest IC after looking would be optimization on the same sample.

## Decision-oriented classifications from retained evidence

### Robust-looking observations

- Archive/reproduction integrity and denominator visibility are reproducible.
- Aggregate momentum rank association is positive across horizons, and non-overlapping offsets retain the sign—but the accepted decision classification is **DATA-CONFOUNDED** because coverage/calendar effects dominate interpretation.
- Realized-volatility rank is negative in aggregate and classified **PROMISING only as a conditioning/risk variable**.

“Robust-looking” means internally consistent under retained checks, not a deployable alpha claim.

### Weak or inconclusive

- Momentum is weak in the early half and concentrated after 2023; the Q5 tail is below Q4 at every horizon.
- Five-session return/pullback is classified **NOT SUPPORTED**; positive-trend pullback diagnostics do not show a stable premium.
- Relative volume is classified **WEAK** despite positive longer-horizon associations.

### Data-confounded or possible artifacts

- Coverage/calendar confounding: 57/60 results can change sign and shape, but coverage moves with unresolved exits and time.
- Vendor adjustment semantics are unverified; membership announcement-time completeness is not established.
- Overlapping labels, roughly five years of sessions, cross-security dependence, and multiple testing limit inference.

### Hypotheses worth testing later

- The final decision-oriented max-close proximity is classified **PROMISING** after controlling for momentum, but needs independently verified adjustment/action data. The earlier strict-high variant is a separate retained diagnostic definition.
- Why momentum concentrates after 2023, using prespecified regimes and new evidence.
- Relative-volume and low-volatility associations with sector/size/liquidity controls and clean coverage.

### Unsafe to act on

- A selected horizon, the 2024 episode, the Q5−Q1 diagnostic, exclusion of low-coverage dates, or any missing-member counterfactual.
- The stronger alternative label anchor, selected after inspection.

## What selected tests prove—and do not prove

| Evidence | Condition | Catches | Passing proves | Does not prove |
|---|---|---|---|---|
| Phase A archive integrity | archived source/hashes parse and match | checkout drift mistaken for corruption | retained run is internally intact | economic truth |
| feature lag/lookback tests | exact shifts and rolling endpoints | off-by-one leakage | formulas match frozen convention | source adjustments are correct |
| label alignment/missing endpoint | exact `t+h`, no fill | accidental forward fill | null/timing semantics | same-close executability |
| identity/universe tests | validity boundaries and five exits | denominator shrinkage | official missing states remain visible | their unobserved returns |
| reproducibility boundaries | output roots/artifact set/hashes | path escape or undeclared artifacts | retained artifacts are controlled | findings generalize |

## Safe to rely on now

- The frozen formulas, point-in-time eligibility mechanics, exact label convention, artifact hashes, and reported sample classifications.

## Usable with documented caveats

- Aggregate diagnostic associations as hypotheses, with explicit coverage, temporal, dependence, and adjustment caveats.

## Not implemented or not safe to rely on

- A strategy return, proof of alpha, portfolio construction rule, optimized horizon, causal explanation, or clean missing-member performance estimate.